## 1 简介

本次主要通过使用```Scikit-Plot```的模块来介绍机器学习的相关可视化，```Scikit-Plot```主要包括以下几个部分：  
* estimators：用于绘制各种算法
* metrics：用于绘制机器学习的onfusion matrix, ROC AUC curves, precision-recall curves等曲线
* cluster：主要用于绘制聚类
* decomposition：主要用于绘制PCA降维

```Scikit-Plot```的地址：https://github.com/reiinakano/scikit-plot

```Scikit-Plot```的官方文档：https://scikit-plot.readthedocs.io/en/stable/

In [ ]:
# --- 机器学习可视化报告：使用 Scikit-Plot 展示各种评估图表 ---
# Scikit-Plot 是 scikit-learn 的扩展可视化库，提供一键生成ML评估图表的功能
import scikitplot as skplt

import sklearn
from sklearn.datasets import load_digits, load_boston, load_breast_cancer
from sklearn.model_selection import train_test_split

from sklearn.ensemble import RandomForestClassifier, RandomForestRegressor, GradientBoostingClassifier, ExtraTreesClassifier
from sklearn.linear_model import LinearRegression, LogisticRegression
from sklearn.cluster import KMeans
from sklearn.decomposition import PCA

import matplotlib.pyplot as plt
import sys

print("Scikit Plot Version : ", skplt.__version__)
print("Scikit Learn Version : ", sklearn.__version__)
print("Python Version : ", sys.version)

## 2 加载数据集

### 2.1 手写数据集

In [ ]:
# 加载手写数字数据集：1797张8x8灰度图像，10个类别(0-9)
digits = load_digits()
X_digits, Y_digits = digits.data, digits.target  # (1797, 64) 特征矩阵

print("Digits Dataset Size : ", X_digits.shape, Y_digits.shape)

# 划分训练集(80%)和测试集(20%)，stratify保证各类别比例一致
X_digits_train, X_digits_test, Y_digits_train, Y_digits_test = train_test_split(X_digits, Y_digits,
                                                                                train_size=0.8,
                                                                                stratify=Y_digits,
                                                                                random_state=1)

print("Digits Train/Test Sizes : ",X_digits_train.shape, X_digits_test.shape, Y_digits_train.shape, Y_digits_test.shape)

### 2.2 肿瘤数据集

In [ ]:
# 加载乳腺癌数据集：569个样本，30个特征，二分类（良性/恶性）
cancer = load_breast_cancer()
X_cancer, Y_cancer = cancer.data, cancer.target

print("Feautre Names : ", cancer.feature_names)
print("Cancer Dataset Size : ", X_cancer.shape, Y_cancer.shape)
# stratify=Y_cancer 确保训练/测试集中良性与恶性样本比例一致
X_cancer_train, X_cancer_test, Y_cancer_train, Y_cancer_test = train_test_split(X_cancer, Y_cancer,
                                                                                train_size=0.8,
                                                                                stratify=Y_cancer,
                                                                                random_state=1)

print("Cancer Train/Test Sizes : ",X_cancer_train.shape, X_cancer_test.shape, Y_cancer_train.shape, Y_cancer_test.shape)

### 2.3 波斯顿房价数据集

In [ ]:
# 加载波士顿房价数据集：506个样本，13个特征，回归任务（预测房价中位数）
boston = load_boston()
X_boston, Y_boston = boston.data, boston.target

print("Boston Dataset Size : ", X_boston.shape, Y_boston.shape)
print("Boston Dataset Features : ", boston.feature_names)
X_boston_train, X_boston_test, Y_boston_train, Y_boston_test = train_test_split(X_boston, Y_boston,
                                                                                train_size=0.8,
                                                                                random_state=1)

print("Boston Train/Test Sizes : ",X_boston_train.shape, X_boston_test.shape, Y_boston_train.shape, Y_boston_test.shape)

## 3 性能可视化  

### 3.1 交叉验证绘制

In [ ]:
# --- 学习曲线（Learning Curve）：展示训练集大小与模型性能的关系 ---
# 学习曲线用于诊断过拟合/欠拟合：训练和验证曲线趋于收敛说明模型已充分学习
# Logistic 回归分类学习曲线
skplt.estimators.plot_learning_curve(LogisticRegression(), X_digits, Y_digits,
                                     cv=7, shuffle=True, scoring="accuracy",
                                     n_jobs=-1, figsize=(6,4), title_fontsize="large", text_fontsize="large",
                                     title="Digits Classification Learning Curve")
plt.show()

# 线性回归回归学习曲线（使用 R^2 评分）
skplt.estimators.plot_learning_curve(LinearRegression(), X_boston, Y_boston,
                                     cv=7, shuffle=True, scoring="r2", n_jobs=-1,
                                     figsize=(6,4), title_fontsize="large", text_fontsize="large",
                                     title="Boston Regression Learning Curve ");
plt.show()

### 3.2 重要性特征绘制

In [ ]:
# --- 特征重要性（Feature Importance）：展示各特征对预测结果的贡献 ---
# 随机森林回归器：通过计算每个特征在各树中的平均不纯度减少量来衡量重要性
rf_reg = RandomForestRegressor()
rf_reg.fit(X_boston_train, Y_boston_train)
print(rf_reg.score(X_boston_test, Y_boston_test))
# 梯度提升分类器：同样可计算特征重要性
gb_classif = GradientBoostingClassifier()
gb_classif.fit(X_cancer_train, Y_cancer_train)
print(gb_classif.score(X_cancer_test, Y_cancer_test))

fig = plt.figure(figsize=(15,6))
ax1 = fig.add_subplot(121)
skplt.estimators.plot_feature_importances(rf_reg, feature_names=boston.feature_names,
                                         title="Random Forest Regressor Feature Importance",
                                         x_tick_rotation=90, order="ascending",
                                         ax=ax1);
ax2 = fig.add_subplot(122)
skplt.estimators.plot_feature_importances(gb_classif, feature_names=cancer.feature_names,
                                         title="Gradient Boosting Classifier Feature Importance",
                                         x_tick_rotation=90,
                                         ax=ax2);
plt.tight_layout()
plt.show()

## 4 机器学习度量(metrics)

### 4.1 混淆矩阵(Confusion Matrix)

In [ ]:
# --- 混淆矩阵（Confusion Matrix）：展示预测结果与真实标签的对应关系 ---
# 对角线元素为正确预测数，非对角线为各类别的误分类情况
log_reg = LogisticRegression()
log_reg.fit(X_digits_train, Y_digits_train)
log_reg.score(X_digits_test, Y_digits_test)
Y_test_pred = log_reg.predict(X_digits_test)

fig = plt.figure(figsize=(15,6))
# 左侧：绝对数量混淆矩阵
ax1 = fig.add_subplot(121)
skplt.metrics.plot_confusion_matrix(Y_digits_test, Y_test_pred,
                                    title="Confusion Matrix",
                                    cmap="Oranges",
                                    ax=ax1)
# 右侧：归一化混淆矩阵（每行之和为1，展示各类别的预测比例）
ax2 = fig.add_subplot(122)
skplt.metrics.plot_confusion_matrix(Y_digits_test, Y_test_pred,
                                    normalize=True,
                                    title="Confusion Matrix",
                                    cmap="Purples",
                                    ax=ax2);
plt.show()

### 4.2 ROC、AUC曲线

In [ ]:
# --- ROC 曲线和 AUC 值 ---
# ROC 曲线：以假正率(FPR)为横轴、真正率(TPR)为纵轴，展示不同阈值下的分类性能
# AUC（曲线下面积）：越接近1.0越好，0.5表示随机猜测
# predict_proba 获取每个类别的预测概率，用于绘制多分类 ROC
Y_test_probs = log_reg.predict_proba(X_digits_test)
skplt.metrics.plot_roc_curve(Y_digits_test, Y_test_probs,
                       title="Digits ROC Curve", figsize=(12,6))
plt.show()

### 4.3 PR曲线

In [ ]:
# --- PR 曲线（Precision-Recall Curve）：精确率-召回率曲线 ---
# PR 曲线适合不平衡数据集：关注正类的精确率和召回率之间的权衡
# 曲线越靠近右上角，模型性能越好
skplt.metrics.plot_precision_recall_curve(Y_digits_test, Y_test_probs,
                       title="Digits Precision-Recall Curve", figsize=(12,6))
plt.show()

### 4.4 轮廓分析（Silhouette Analysis）

In [ ]:
# --- 轮廓分析（Silhouette Analysis）：评估聚类质量 ---
# 轮廓系数 s(i) = (b(i) - a(i)) / max(a(i), b(i))
# a(i)：样本 i 到同簇其他样本的平均距离（簇内紧密度）
# b(i)：样本 i 到最近邻簇所有样本的平均距离（簇间分离度）
# 轮廓系数范围 [-1, 1]：越接近1说明聚类越好
kmeans = KMeans(n_clusters=10, random_state=1)
kmeans.fit(X_digits_train, Y_digits_train)
cluster_labels = kmeans.predict(X_digits_test)
skplt.metrics.plot_silhouette(X_digits_test, cluster_labels,
                              figsize=(8,6))
plt.show()

### 4.5 可靠性曲线（Calibration Curve ，Reliability Curves）

In [ ]:
# --- 可靠性曲线（Calibration Curve）：评估模型预测概率的校准程度 ---
# 理想情况下，预测概率为0.8的样本中应有80%为正类
# 对角线表示完美校准，偏离对角线越远说明概率估计越不准确
lr_probas = LogisticRegression().fit(X_cancer_train, Y_cancer_train).predict_proba(X_cancer_test)
rf_probas = RandomForestClassifier().fit(X_cancer_train, Y_cancer_train).predict_proba(X_cancer_test)
gb_probas = GradientBoostingClassifier().fit(X_cancer_train, Y_cancer_train).predict_proba(X_cancer_test)
et_scores = ExtraTreesClassifier().fit(X_cancer_train, Y_cancer_train).predict_proba(X_cancer_test)

probas_list = [lr_probas, rf_probas, gb_probas, et_scores]
clf_names = ['Logistic Regression', 'Random Forest', 'Gradient Boosting', 'Extra Trees Classifier']
skplt.metrics.plot_calibration_curve(Y_cancer_test,
                                     probas_list,
                                     clf_names, n_bins=15,
                                     figsize=(12,6)
                                     )
plt.show()

### 4.6 KS检验 

In [ ]:
# --- KS 检验（Kolmogorov-Smirnov Statistic）：评估二分类模型的区分能力 ---
# KS 统计量 = max|F_1(x) - F_0(x)|，即正类和负类累积分布的最大差异
# KS 值越大说明模型区分正负样本的能力越强
rf = RandomForestClassifier()
rf.fit(X_cancer_train, Y_cancer_train)
Y_cancer_probas = rf.predict_proba(X_cancer_test)
skplt.metrics.plot_ks_statistic(Y_cancer_test, Y_cancer_probas, figsize=(10,6))
plt.show()

### 4.7 累积收益曲线

In [ ]:
# --- 累积收益曲线（Cumulative Gains Chart）：评估模型在不同采样比例下捕获正类的能力
# 曲线越陡峭，说明模型能在较少的样本中捕获更多的正类
skplt.metrics.plot_cumulative_gain(Y_cancer_test, Y_cancer_probas, figsize=(10,6))
plt.show()

### 4.8 Lift 曲线

In [ ]:
# --- Lift 曲线：衡量模型相比随机选择的提升倍数 ---
# Lift = (在前x%样本中捕获的正类比例) / (正类在总体中的比例)
# Lift > 1 表示模型优于随机选择
skplt.metrics.plot_lift_curve(Y_cancer_test, Y_cancer_probas, figsize=(10,6))
plt.show()

## 5 聚类方法

### 5.1 手肘法（Elbow Method）

In [ ]:
# --- 手肘法（Elbow Method）：确定最佳聚类数 K ---
# 绘制不同 K 值下的簇内平方误差和（SSE/inertia）
# 拐点处（"肘部"）对应的 K 值通常为最佳聚类数
# 原理：K 增大时 SSE 必然下降，但下降速率会在某一点明显变缓
skplt.cluster.plot_elbow_curve(KMeans(random_state=1),
                               X_digits,
                               cluster_ranges=range(2, 20),
                               figsize=(8,6))
plt.show()

## 6 降维方法

### 6.1 PCA

In [ ]:
# --- PCA 方差解释比例：展示每个主成分解释的方差占比 ---
# 帮助决定保留多少主成分（通常选择累计方差占比达到85%-95%的主成分数）
pca = PCA(random_state=1)
pca.fit(X_digits)
skplt.decomposition.plot_pca_component_variance(pca, figsize=(8,6))
plt.show()

### 6.2 2-D Projection

In [ ]:
# --- PCA 二维投影：将高维数据投影到前2个主成分进行可视化 ---
# 直观展示数据在主成分空间中的分布和各类别的可分性
skplt.decomposition.plot_pca_2d_projection(pca, X_digits, Y_digits,
                                           figsize=(10,10),
                                           cmap="tab10")
plt.show()